# CAGP on FB15k-237 (High Diversity)

**Goal:** Confirm CAGP maintains high performance on relation-rich KGs.

**Dataset:** FB15k-237 (237 relations)

**Expected:** CAGP ≈ VanillaGPKGE ≈ 0.85+ AUROC (both should work well)

In [1]:
!pip install -q torch numpy scikit-learn matplotlib pykeen

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cpu


In [3]:
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,
    'batch_size': 1024,
    'lr': 0.001,
    'kl_weight': 0.01,
    'seeds': [42, 123, 456],
}

In [4]:
def load_fb15k237():
    from pykeen.datasets import FB15k237
    dataset = FB15k237(create_inverse_triples=False)

    def extract_triples(tf):
        return [(h, r, t) for h, r, t in tf.triples]

    train = extract_triples(dataset.training)
    valid = extract_triples(dataset.validation)
    test = extract_triples(dataset.testing)

    entities = set()
    relations = set()
    for h, r, t in train + valid + test:
        entities.add(h)
        entities.add(t)
        relations.add(r)

    return {
        'train': train, 'valid': valid, 'test': test,
        'entities': entities, 'relations': relations,
    }

data = load_fb15k237()
print(f"FB15k-237: {len(data['train'])} train, {len(data['test'])} test")
print(f"Entities: {len(data['entities'])}, Relations: {len(data['relations'])}")

INFO:pykeen.utils:Using opt_einsum
INFO:pykeen.datasets.base:downloading data from https://download.microsoft.com/download/8/7/0/8700516A-AB3D-4850-B4BB-805C515AECE1/FB15K-237.2.zip to /root/.data/pykeen/datasets/fb15k237/FB15K-237.2.zip


FB15k-237: 272115 train, 20438 test
Entities: 14505, Relations: 237


In [5]:
class DistMult(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails):
        h = self.entity_emb(heads)
        r = self.relation_emb(relations)
        t = self.entity_emb(tails)
        return (h * r * t).sum(dim=-1)

    def get_uncertainty(self, heads, relations, tails):
        scores = torch.sigmoid(self.forward(heads, relations, tails))
        uncertainty = -scores * torch.log(scores + 1e-10) - (1-scores) * torch.log(1-scores + 1e-10)
        return uncertainty


class VanillaGPKGE(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.num_entities = num_entities
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

    def forward(self, heads, relations, tails, use_sampling=True):
        if use_sampling and self.training:
            h = self._sample(heads)
            t = self._sample(tails)
        else:
            h = self.entity_mean[heads]
            t = self.entity_mean[tails]
        r = self.relation_emb(relations)
        return (h * r * t).sum(dim=-1)

    def _sample(self, indices):
        mean = self.entity_mean[indices]
        std = torch.exp(0.5 * self.entity_logvar[indices])
        return mean + std * torch.randn_like(std)

    def get_uncertainty(self, heads, relations, tails):
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        return (h_var + t_var) / 2

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities


class CoverageAugmentedGPKGE(nn.Module):
    def __init__(self, num_entities, num_relations, dim, initial_alpha=0.5, learn_alpha=True):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations

        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)

        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

        if learn_alpha:
            self.alpha_logit = nn.Parameter(torch.logit(torch.tensor(initial_alpha)))
        else:
            self.register_buffer('alpha_logit', torch.logit(torch.tensor(initial_alpha)))

    def forward(self, heads, relations, tails, use_sampling=True):
        if use_sampling and self.training:
            h = self._sample(heads)
            t = self._sample(tails)
        else:
            h = self.entity_mean[heads]
            t = self.entity_mean[tails]
        r = self.relation_emb(relations)
        return (h * r * t).sum(dim=-1)

    def _sample(self, indices):
        mean = self.entity_mean[indices]
        std = torch.exp(0.5 * self.entity_logvar[indices])
        return mean + std * torch.randn_like(std)

    def get_uncertainty(self, heads, relations, tails):
        h_var = torch.exp(self.entity_logvar[heads]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[tails]).mean(dim=-1)
        gp_var = (h_var + t_var) / 2

        h_seen = self.coverage[heads, relations]
        t_seen = self.coverage[tails, relations]
        coverage_unc = 2.0 - h_seen - t_seen

        gp_var_norm = gp_var / (gp_var.mean() + 1e-8) * (coverage_unc.mean() + 1e-8)

        alpha = torch.sigmoid(self.alpha_logit)
        return alpha * gp_var_norm + (1 - alpha) * coverage_unc

    def precompute_coverage(self, triples, entity_to_idx, relation_to_idx):
        for h, r, t in triples:
            self.coverage[entity_to_idx[h], relation_to_idx[r]] = 1.0
            self.coverage[entity_to_idx[t], relation_to_idx[r]] = 1.0

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities

    def get_alpha(self):
        return torch.sigmoid(self.alpha_logit).item()


print("Models defined")

Models defined


In [6]:
def train_model(model, triples, entity_to_idx, relation_to_idx, epochs, is_gp=False):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.BCEWithLogitsLoss()

    if hasattr(model, 'precompute_coverage'):
        model.precompute_coverage(triples, entity_to_idx, relation_to_idx)

    heads = torch.tensor([entity_to_idx[h] for h, r, t in triples])
    relations = torch.tensor([relation_to_idx[r] for h, r, t in triples])
    tails = torch.tensor([entity_to_idx[t] for h, r, t in triples])

    num_entities = len(entity_to_idx)
    dataset = TensorDataset(heads, relations, tails)
    loader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True)

    model.train()
    for epoch in range(epochs):
        for batch_h, batch_r, batch_t in loader:
            batch_h, batch_r, batch_t = batch_h.to(device), batch_r.to(device), batch_t.to(device)

            if is_gp:
                pos_scores = model(batch_h, batch_r, batch_t, use_sampling=True)
            else:
                pos_scores = model(batch_h, batch_r, batch_t)

            neg_t = torch.randint(0, num_entities, batch_t.shape, device=device)
            if is_gp:
                neg_scores = model(batch_h, batch_r, neg_t, use_sampling=True)
            else:
                neg_scores = model(batch_h, batch_r, neg_t)

            loss = criterion(pos_scores, torch.ones_like(pos_scores)) + \
                   criterion(neg_scores, torch.zeros_like(neg_scores))

            if is_gp and hasattr(model, 'kl_loss'):
                loss += CONFIG['kl_weight'] * model.kl_loss()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return model


def evaluate_auroc(model, test_triples, entity_to_idx, relation_to_idx):
    model.eval()
    heads = torch.tensor([entity_to_idx.get(h, 0) for h, r, t in test_triples]).to(device)
    relations = torch.tensor([relation_to_idx.get(r, 0) for h, r, t in test_triples]).to(device)
    tails = torch.tensor([entity_to_idx.get(t, 0) for h, r, t in test_triples]).to(device)

    with torch.no_grad():
        id_unc = model.get_uncertainty(heads, relations, tails).cpu().numpy()
        neg_tails = torch.randint(0, len(entity_to_idx), tails.shape, device=device)
        ood_unc = model.get_uncertainty(heads, relations, neg_tails).cpu().numpy()

    labels = np.concatenate([np.ones(len(id_unc)), np.zeros(len(ood_unc))])
    scores = np.concatenate([-id_unc, -ood_unc])
    return roc_auc_score(labels, scores)

In [7]:
ent2idx = {e: i for i, e in enumerate(data['entities'])}
rel2idx = {r: i for i, r in enumerate(data['relations'])}

results = {'DistMult': [], 'VanillaGPKGE': [], 'CAGP': []}
alphas = []

for seed in CONFIG['seeds']:
    print(f"\n{'='*50}")
    print(f"Seed {seed}")
    print('='*50)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # DistMult
    dm = DistMult(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    dm = train_model(dm, data['train'], ent2idx, rel2idx, CONFIG['epochs'], is_gp=False)
    auroc_dm = evaluate_auroc(dm, data['test'], ent2idx, rel2idx)
    results['DistMult'].append(auroc_dm)
    print(f"  DistMult AUROC: {auroc_dm:.4f}")

    # Vanilla GP-KGE
    gp = VanillaGPKGE(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    gp = train_model(gp, data['train'], ent2idx, rel2idx, CONFIG['epochs'], is_gp=True)
    auroc_gp = evaluate_auroc(gp, data['test'], ent2idx, rel2idx)
    results['VanillaGPKGE'].append(auroc_gp)
    print(f"  VanillaGPKGE AUROC: {auroc_gp:.4f}")

    # CAGP
    cagp = CoverageAugmentedGPKGE(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
    cagp = train_model(cagp, data['train'], ent2idx, rel2idx, CONFIG['epochs'], is_gp=True)
    auroc_cagp = evaluate_auroc(cagp, data['test'], ent2idx, rel2idx)
    results['CAGP'].append(auroc_cagp)
    alphas.append(cagp.get_alpha())
    print(f"  CAGP AUROC: {auroc_cagp:.4f} (α={cagp.get_alpha():.3f})")


Seed 42
  DistMult AUROC: 0.1788
  VanillaGPKGE AUROC: 0.7484
  CAGP AUROC: 0.9595 (α=0.500)

Seed 123
  DistMult AUROC: 0.1818
  VanillaGPKGE AUROC: 0.7480
  CAGP AUROC: 0.9595 (α=0.500)

Seed 456
  DistMult AUROC: 0.1857
  VanillaGPKGE AUROC: 0.7502
  CAGP AUROC: 0.9593 (α=0.500)


In [8]:
print("\n" + "="*60)
print("FB15k-237 RESULTS (237 relations)")
print("="*60)
print(f"{'Model':<20} {'AUROC':<15} {'vs VanillaGPKGE'}")
print("-"*60)

gp_mean = np.mean(results['VanillaGPKGE'])
for model_name in ['DistMult', 'VanillaGPKGE', 'CAGP']:
    mean = np.mean(results[model_name])
    std = np.std(results[model_name])
    delta = mean - gp_mean
    print(f"{model_name:<20} {mean:.4f} ± {std:.3f}   {delta:+.4f}")

print("-"*60)
print(f"Learned α = {np.mean(alphas):.3f} ± {np.std(alphas):.3f}")

cagp_mean = np.mean(results['CAGP'])
if cagp_mean >= gp_mean - 0.02:
    print("\nSUCCESS: CAGP maintains performance on FB15k-237!")
else:
    print("\nWARNING: CAGP underperforms on FB15k-237")


FB15k-237 RESULTS (237 relations)
Model                AUROC           vs VanillaGPKGE
------------------------------------------------------------
DistMult             0.1821 ± 0.003   -0.5668
VanillaGPKGE         0.7489 ± 0.001   +0.0000
CAGP                 0.9595 ± 0.000   +0.2106
------------------------------------------------------------
Learned α = 0.500 ± 0.000

SUCCESS: CAGP maintains performance on FB15k-237!


In [9]:
import json
output = {
    'dataset': 'FB15k-237',
    'num_relations': len(data['relations']),
    'results': {m: {'mean': float(np.mean(results[m])), 'std': float(np.std(results[m]))} for m in results},
    'learned_alpha': {'mean': float(np.mean(alphas)), 'std': float(np.std(alphas))}
}
with open('cagp_fb15k237_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print("Results saved")

Results saved
